# Query and Generate Query Answers using Vertex AI Search
- May 2026
- Numantic Solutions (numanticsolutions.com)

In [1]:
import os, sys
import json
import time

# Pandas and Pandas Google Big Query
import pandas as pd
import pandas_gbq as pbq

# Vertex AI search and generate class
import vai_search_app_query as vsaq

# Numantic utilities
utils_path = "../utils"
sys.path.insert(0, utils_path)
from utils import ApiAuthentication
api_configs = ApiAuthentication(client="Numantic")


## Read test data

In [2]:
input_data_path = "../data/rag_eval_dataset"
multi_pas_qs = "multi_passage_answer_questions.csv"
no_answer_qs = "no_answer_questions.csv"
single_pas_answer_qs = "single_passage_answer_questions.csv"

df_spqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, single_pas_answer_qs))
# df_spqs.head()


## Read document data

In [3]:
gcp_project_id = os.environ["GOOGLE_CLOUD_PROJECT_ID"]
docs_table = "ns_bq.rag_tests_2"

df_docs = pbq.read_gbq(query_or_table=docs_table,
                  project_id=gcp_project_id)

Downloading: 100%|██████████|


## Test the single-passage questions to establish baseline RAG performance

In [4]:
search_app_id = "rag-tests-searchapp-v26"
gcp_project_number = "643773332888"
gcp_location = "global"


In [11]:
# Run through each single passage question
result_rows = []
for idx in df_spqs.index:

    # Create a query object
    test_querier = vsaq.QueryVaiSearch(search_app_id=search_app_id,
                                       gcp_project=gcp_project_number,
                                       gcp_location=gcp_location)

    # Create a query and
    query = df_spqs.loc[idx, "question"]
    test_querier.search_and_generate_answer(query=query)

    # Check if correct source found
    if len(test_querier.res_doc_ids) > 0 and \
            "doc_{}".format(df_spqs.loc[idx, "document_index"]) in test_querier.res_doc_ids:
        found_correct_source = True
    else:
        found_correct_source = False

    sum_dict = dict(query=query,
                    rag_answer=test_querier.answer,
                    answer=df_spqs.loc[idx, "answer"],
                    rag_citations=test_querier.res_doc_ids,
                    source_document_index="doc_{}.txt".format(df_spqs.loc[idx, "document_index"]),
                    found_correct_source=found_correct_source
                    )

    result_rows.append(sum_dict)

    time.sleep(5)

# Put results into a dataframe
df_results = pd.DataFrame(data=result_rows)


## Examine test results

In [12]:
# Dataframe with results
df_results.head()
df_results


,query,rag_answer,answer,rag_citations,source_document_index,found_correct_source
0,What do keybullet kin drop?,Keybullet Kin drop a key upon death. If a Keyb...,Keybullet kin drop a key upon death.,[doc_0],doc_0.txt,True
1,What kind of gun does the bandana bullet kin use?,A summary could not be generated for your sear...,The bandana bullet kin wields a machine pistol.,[doc_0],doc_0.txt,True
2,What do the giants look like?,"There are several giants. One giant is burly, ...","One giant is burly, grey-skinned, and 20 feet ...","[doc_1, doc_15, doc_0, doc_11, doc_12, doc_8, ...",doc_1.txt,True
3,What happens on day 2?,"After a few miles of winding tunnel, you emerg...","After a few miles of winding tunnel, you emerg...","[doc_1, doc_17, doc_9, doc_16, doc_14, doc_8, ...",doc_1.txt,True
4,What were the requirements for the project?,"For the vector DB, the main requirements were ...",The tool had the following requirements:\n- Ch...,"[doc_2, doc_11, doc_17, doc_3]",doc_2.txt,True
5,What data did was used to test the prototype?,A summary could not be generated for your sear...,Grace Hopper's Wikipedia page and Alan Turing'...,[doc_2],doc_2.txt,True
6,How do the data storage options compare?,Data storage options involve a hierarchy of me...,For fast start: use SQLite3 and ChromaDB (File...,"[doc_11, doc_16, doc_7]",doc_3.txt,False
7,When was UTF-8 support added for European lang...,UTF-8 encoding for European languages was incl...,UTF-8 support was added for European languages...,"[doc_3, doc_12]",doc_3.txt,True
8,How do I make a button?,"To make a button, you can use `marimo` (mo).\n...",import marimo as mo\n\nbutton = mo.ui.run_butt...,"[doc_4, doc_16, doc_15]",doc_4.txt,True
9,When might I use caching?,Caching can be used to cache expensive computa...,"You might use caching when, for example, your ...","[doc_4, doc_11, doc_7]",doc_4.txt,True


## Manually review RAG responses and identify questions RAG wasn't able to answer

In [8]:
no_ans_idx = [5, 6, 10, 11, 13, 19, 26, 35, 37]

df_results.loc[no_ans_idx]

,query,rag_answer,answer,rag_citations,source_document_index,found_correct_source
5,What data did was used to test the prototype?,A summary could not be generated for your sear...,Grace Hopper's Wikipedia page and Alan Turing'...,[doc_2],doc_2.txt,True
6,How do the data storage options compare?,The data storage options in GPUs involve a mem...,For fast start: use SQLite3 and ChromaDB (File...,"[doc_11, doc_16, doc_7]",doc_3.txt,False
10,What are the key topics of this article?,The key topics of the provided articles includ...,"The key topics of this article are: ""why prior...","[doc_19, doc_17, doc_3, doc_11, doc_10, doc_7]",doc_5.txt,False
11,Do I have to do something all by myself to be ...,"To be acknowledged for something, you do not n...",You don’t have to be solely responsible for so...,"[doc_8, doc_2, doc_11, doc_7, doc_10, doc_16, ...",doc_5.txt,False
13,"What kinds of AI carry ""systematic risks""?",A summary could not be generated for your sear...,"For now, general purpose AI models that were t...",[],doc_6.txt,False
19,How do the people who commit atrocious acts an...,A summary could not be generated for your sear...,The Zone of Interest does not really different...,[doc_7],doc_9.txt,False
26,Why are the top-right values in the raw attent...,A summary could not be generated for your sear...,A mask of negative infinity is applied to the ...,[doc_13],doc_13.txt,True
35,When is this game set?,The game *Alan Wake 2* was released on October...,"This game is set in 2023, thirteen years after...","[doc_17, doc_16, doc_7, doc_8, doc_3]",doc_17.txt,True
37,Who was resurrected with a group of other murd...,"A naked man, later identified as former FBI Ag...",Lou was resurrected along with a handful of ot...,[doc_17],doc_18.txt,False


## Add a Filter to the Vertex AI search and rerun single-passage tests

In [7]:
# Add a filter and test questions RAG had trouble with in original run
result_rows = []
for idx in df_spqs.index:

    # Create a query object
    test_querier = vsaq.QueryVaiSearch(search_app_id=search_app_id,
                                       gcp_project=gcp_project_number,
                                       gcp_location=gcp_location)

    # Create a query and
    query = df_spqs.loc[idx, "question"]

    # Find the source URL which is indexable
    doc_index = "doc_{}".format(df_spqs.loc[idx, "document_index"])
    search_filter = f'doc_index: ANY("{doc_index}")'

    # Search with a filter
    test_querier.search_and_generate_answer(query=query,
                                            search_filter=search_filter)

    # Check if correct source found
    if len(test_querier.res_doc_ids) > 0 and \
            "doc_{}".format(df_spqs.loc[idx, "document_index"]) in test_querier.res_doc_ids:
        found_correct_source = True
    else:
        found_correct_source = False

    sum_dict = dict(query=query,
                    rag_answer=test_querier.answer,
                    answer=df_spqs.loc[idx, "answer"],
                    rag_citations=test_querier.res_doc_ids,
                    source_document_index="doc_{}.txt".format(df_spqs.loc[idx, "document_index"]),
                    found_correct_source=found_correct_source
                    )

    result_rows.append(sum_dict)

    time.sleep(5)

# Put results into a dataframe
df_results_f = pd.DataFrame(data=result_rows)


In [8]:
# View results
df_results_f


,query,rag_answer,answer,rag_citations,source_document_index,found_correct_source
0,What do keybullet kin drop?,Keybullet Kin run away from the player. Upon d...,Keybullet kin drop a key upon death.,[doc_0],doc_0.txt,True
1,What kind of gun does the bandana bullet kin use?,A summary could not be generated for your sear...,The bandana bullet kin wields a machine pistol.,[doc_0],doc_0.txt,True
2,What do the giants look like?,"There are several giants. One giant is burly, ...","One giant is burly, grey-skinned, and 20 feet ...",[doc_1],doc_1.txt,True
3,What happens on day 2?,"On Day 2, after traveling a few miles through ...","After a few miles of winding tunnel, you emerg...",[doc_1],doc_1.txt,True
4,What were the requirements for the project?,"The project, STICI-note, is a locally run RAG ...",The tool had the following requirements:\n- Ch...,[doc_2],doc_2.txt,True
5,What data did was used to test the prototype?,A summary could not be generated for your sear...,Grace Hopper's Wikipedia page and Alan Turing'...,[doc_2],doc_2.txt,True
6,How do the data storage options compare?,A summary could not be generated for your sear...,For fast start: use SQLite3 and ChromaDB (File...,[],doc_3.txt,False
7,When was UTF-8 support added for European lang...,UTF-8 encoding for European languages was incl...,UTF-8 support was added for European languages...,[doc_3],doc_3.txt,True
8,How do I make a button?,"To make a button, you need to import the `mari...",import marimo as mo\n\nbutton = mo.ui.run_butt...,[doc_4],doc_4.txt,True
9,When might I use caching?,Caching can be used to cache expensive computa...,"You might use caching when, for example, your ...",[doc_4],doc_4.txt,True


## Feed source document to AI client and rerun single-passage tests

In [9]:
# Use a direct AI client query with source text to test questions RAG had trouble with in original run
result_rows = []
for idx in df_spqs.index:

    # Create a query object
    test_querier = vsaq.QueryVaiSearch(search_app_id=search_app_id,
                                       gcp_project=gcp_project_number,
                                       gcp_location=gcp_location)

    # Create a query and
    query = df_spqs.loc[idx, "question"]

    # Get source content
    doc_index = "doc_{}".format(df_spqs.loc[idx, "document_index"])
    mask = df_docs["doc_index"] == doc_index
    idxd0 = df_docs[mask].index[0]

    # Search with a filter
    test_querier.query_ai_client(query=query,
                                 source_text=df_docs.loc[idxd0, "content"])

    # All sources are correct since we're feeding the model source text
    found_correct_source = True

    sum_dict = dict(query=query,
                    rag_answer=test_querier.client_response.text,
                    answer=df_spqs.loc[idx, "answer"],
                    rag_citations=[doc_index],
                    source_document_index="doc_{}.txt".format(df_spqs.loc[idx, "document_index"]),
                    found_correct_source=found_correct_source
                    )

    result_rows.append(sum_dict)

    time.sleep(5)

# Put results into a dataframe
df_results_c = pd.DataFrame(data=result_rows)


In [10]:
# View results
df_results_c


,query,rag_answer,answer,rag_citations,source_document_index,found_correct_source
0,What do keybullet kin drop?,Keybullet Kin drop a key.,Keybullet kin drop a key upon death.,[doc_0],doc_0.txt,True
1,What kind of gun does the bandana bullet kin use?,Bandana Bullet Kin wield Machine Pistols.,The bandana bullet kin wields a machine pistol.,[doc_0],doc_0.txt,True
2,What do the giants look like?,"One giant is burly, grey-skinned, 20 feet tall...","One giant is burly, grey-skinned, and 20 feet ...",[doc_1],doc_1.txt,True
3,What happens on day 2?,"On Day 2, the travelers encounter two Ropers a...","After a few miles of winding tunnel, you emerg...",[doc_1],doc_1.txt,True
4,What were the requirements for the project?,The project requirements were:\n* A chatbot ...,The tool had the following requirements:\n- Ch...,[doc_2],doc_2.txt,True
5,What data did was used to test the prototype?,The prototype was tested using Grace Hopper's ...,Grace Hopper's Wikipedia page and Alan Turing'...,[doc_2],doc_2.txt,True
6,How do the data storage options compare?,Data storage options are categorized by scale ...,For fast start: use SQLite3 and ChromaDB (File...,[doc_3],doc_3.txt,True
7,When was UTF-8 support added for European lang...,UTF-8 encoding for European languages was adde...,UTF-8 support was added for European languages...,[doc_3],doc_3.txt,True
8,How do I make a button?,"To make a button, import `marimo as mo`, then ...",import marimo as mo\n\nbutton = mo.ui.run_butt...,[doc_4],doc_4.txt,True
9,When might I use caching?,You might use caching for expensive intermedia...,"You might use caching when, for example, your ...",[doc_4],doc_4.txt,True
